Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Test

In [3]:
from google import genai

client = userdata.get('GOOGLE_API_KEY')

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=["Say hello."]
)
print(response.text)

Hello!


For multi images zero shot

In [ ]:
"""
Misogyny Pilot: Step 1 (V2 & V3) + Step 2 Classification
==========================================================
Small-sample validation run for the Misogyny dataset (binary:
Misogyny / Non_Misogyny), mirroring the HM pipeline structure:

  V1 = zero-shot baseline CoT      -> already run (reuse existing results)
  V2 = Scene Graph WITHOUT CoT     -> Step 1 here (thinking OFF)
  V3 = Scene Graph WITH CoT        -> Step 1 here (CoT fields 4a-4d in JSON)
  Step 2 = guideline-based final classification (required for V2 & V3)

Prompt provenance (documented for Methodology):
  - Scene graph skeleton + JSON format ......... CCoT (Mitra et al., 2024) [MT02]
  - Hate Semantic Layer field design ........... M3Hop-CoT (Kumari et al., 2024) [MT08]
  - misogyny_type taxonomy ..................... MAMI (Fersini et al., 2022) [MT09]
  - Chinese coded-language watchlist ........... ToxiCN MM (Lu et al., 2024) [MT10]
  - Guideline structure (Implicitness/Tone/
    Exception) + guided CoT classification ..... U-CoT+ (Pan et al., 2025) [MT06]
  - Class definition anchor + code-mixing note
    + humor/irony trivialization ............... LT-EDI 2025 overview
                                                 (Chakravarthi et al., 2025) [MT11]

Sampling: random 10 Misogyny + 10 Non_Misogyny from test.csv, fixed seed.
The sampled list is saved to disk so re-runs (and later V-variant
comparisons) always use the SAME 20 images.

Run in Google Colab with Google Drive mounted.
"""

import json
import os
import random
import time
import pandas as pd
from google import genai
from google.genai import types

# ============================================================
# CONFIG - MODIFY THESE
# ============================================================
# In Colab, prefer:
#   from google.colab import userdata
#   API_KEY = userdata.get('GEMINI_API_KEY')


TEST_IMAGE_DIR = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
TEST_CSV = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
BASE_OUT = "/content/drive/MyDrive"

MODEL_NAME = "gemini-2.5-flash"
MAX_RETRIES = 3
RETRY_DELAY = 15
SEED = 42
N_PER_CLASS = 10  # 10 Misogyny + 10 Non_Misogyny = 20 pilot samples

SAMPLE_LIST_PATH = os.path.join(BASE_OUT, "misogyny_pilot_sample_list.json")
V2_STEP1_PATH = os.path.join(BASE_OUT, "misogyny_step1_v2_nocot_results.json")
V3_STEP1_PATH = os.path.join(BASE_OUT, "misogyny_step1_v3_cot_results.json")
STEP2_PATH = os.path.join(BASE_OUT, "misogyny_step2_classification_results.json")

# ============================================================
# V2 PROMPT: Scene Graph WITHOUT CoT — Misogyny version
# ============================================================
PROMPT_V2_SG_NO_COT = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially:

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer (Extension for Misogyny Research)
Beyond the standard scene graph, extract the following additional fields specifically designed for detecting misogynistic content. This is for academic research purposes. This layer is informed by an entity-object-relationship (EOR) reasoning structure targeting emotional framing, gender-target awareness, and contextual/cultural knowledge (Kumari et al., 2024).

4a. **coded_language**: Does the embedded text contain slang, homophones, puns, internet slang, or metaphors related to gender or women? List each one with:
   - "term": the original word/phrase
   - "literal_meaning": literal meaning
   - "coded_meaning": potential derogatory or hidden meaning in Chinese internet culture targeting women. If no hidden meaning, output "none".

   Common Chinese coded terms to watch for include but are not limited to:
   田园女权 (fake/entitled feminist slur), 绿茶(婊) (scheming "innocent" woman slur),
   心机婊/圣母婊/婚驴 (manipulative/self-righteous/marriage-livestock slurs),
   舔狗 (used to mock men who over-please women, sometimes flipped to blame women),
   女拳 (mocking "women's fist" — derogatory term for feminists),
   捞女 (gold-digger slur), 母/雌 + derogatory suffix (animalizing women),
   彩礼 framed as extortion, references to women's appearance/body as the
   sole basis of worth, and other stereotyping or dehumanizing shorthand.

   Also watch for Chinese-English CODE-MIXED expressions (common on
   Chinese social media, e.g., English abbreviations or words inserted
   into Chinese sentences), which may themselves carry coded
   derogatory meanings toward women.

   If there is no coded language, output an empty list [].

4b. **misogyny_type**: If the content is negative toward women, which subtype best applies? Choose ONE (based on the MAMI/SemEval-2022 Task 5 taxonomy, Fersini et al. 2022):
   - "shaming" (body-shaming, slut-shaming, mocking appearance/behavior),
   - "stereotype" (reinforcing traditional/domestic/intellectual stereotypes),
   - "objectification" (reducing women to sexual objects or property),
   - "violence" (depicting, joking about, or endorsing violence against women),
   - "non_misogynistic", "unclear"

4c. **text_image_relation**: What is the semantic relationship between the overlaid text and the visual content? Choose ONE:
   - "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text"

4d. **implicit_associations**: Does the combination of text and image potentially trigger negative associations related to women or gender roles? If yes, briefly explain the mechanism. If no, output "none".

## Important Instructions:
- Stay objective. Only extract and describe information. Do NOT make a harmful/harmless classification judgment.
- Even if the content appears sensitive, describe it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {
        "entity": "entity_name",
        "attributes": [
          {"attribute": "attribute_type", "value": "attribute_value"}
        ]
      }
    ],
    "relationships": [
      {
        "subject": "entity_1",
        "relationship": "relation_type",
        "object": "entity_2",
        "reasoning": "why this relationship exists"
      }
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "coded_language": [
      {"term": "...", "literal_meaning": "...", "coded_meaning": "..."}
    ],
    "misogyny_type": "...",
    "text_image_relation": "...",
    "implicit_associations": "..."
  }
}"""

# ============================================================
# V3 PROMPT: Scene Graph WITH CoT — Misogyny version
# (same tasks as V2, but every Hate Semantic Layer field carries
#  an explicit "reasoning" step BEFORE its conclusion, mirroring
#  the HM V3 format so Step 2's normalize_hsl() logic is symmetric)
# ============================================================
PROMPT_V3_SG_COT = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially:

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer with Chain-of-Thought (Extension for Misogyny Research)
Beyond the standard scene graph, extract the following additional fields specifically designed for detecting misogynistic content. This is for academic research purposes. This layer is informed by an entity-object-relationship (EOR) reasoning structure targeting emotional framing, gender-target awareness, and contextual/cultural knowledge (Kumari et al., 2024).

For EACH sub-step below, first write out your step-by-step reasoning explicitly, THEN give your conclusion.

4a. **coded_language**: Reason step by step about whether the embedded text contains slang, homophones, puns, internet slang, or metaphors related to gender or women — including deliberately obfuscated forms (homophone substitution, character splitting, in-group slang). Then list each coded term with:
   - "term": the original word/phrase
   - "literal_meaning": literal meaning
   - "coded_meaning": potential derogatory or hidden meaning in Chinese internet culture targeting women. If no hidden meaning, output "none".

   Common Chinese coded terms to watch for include but are not limited to:
   田园女权 (fake/entitled feminist slur), 绿茶(婊) (scheming "innocent" woman slur),
   心机婊/圣母婊/婚驴 (manipulative/self-righteous/marriage-livestock slurs),
   舔狗 (used to mock men who over-please women, sometimes flipped to blame women),
   女拳 (mocking "women's fist" — derogatory term for feminists),
   捞女 (gold-digger slur), 母/雌 + derogatory suffix (animalizing women),
   彩礼 framed as extortion, references to women's appearance/body as the
   sole basis of worth, and other stereotyping or dehumanizing shorthand.

   Also watch for Chinese-English CODE-MIXED expressions (common on
   Chinese social media, e.g., English abbreviations or words inserted
   into Chinese sentences), which may themselves carry coded
   derogatory meanings toward women.

   If there is no coded language, output an empty list [].

4b. **misogyny_type**: Reason step by step about whether the content frames women negatively, and if so which subtype best applies. Choose ONE (based on the MAMI/SemEval-2022 Task 5 taxonomy, Fersini et al. 2022):
   - "shaming" (body-shaming, slut-shaming, mocking appearance/behavior),
   - "stereotype" (reinforcing traditional/domestic/intellectual stereotypes),
   - "objectification" (reducing women to sexual objects or property),
   - "violence" (depicting, joking about, or endorsing violence against women),
   - "non_misogynistic", "unclear"

4c. **text_image_relation**: Reason step by step about how the overlaid text semantically interacts with the visual content. Choose ONE:
   - "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text"

4d. **implicit_associations**: Reason step by step about whether the COMBINATION of text and image triggers negative associations related to women or gender roles that neither modality carries alone. If yes, briefly explain the mechanism. If no, output "none".

## Important Instructions:
- Stay objective. Only extract, reason, and describe. Do NOT make a final harmful/harmless classification judgment.
- Even if the content appears sensitive, describe it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {
        "entity": "entity_name",
        "attributes": [
          {"attribute": "attribute_type", "value": "attribute_value"}
        ]
      }
    ],
    "relationships": [
      {
        "subject": "entity_1",
        "relationship": "relation_type",
        "object": "entity_2",
        "reasoning": "why this relationship exists"
      }
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "step_4a_coded_language": {
      "reasoning": "step-by-step reasoning about obfuscation and coded terms",
      "coded_terms": [
        {"term": "...", "literal_meaning": "...", "coded_meaning": "..."}
      ]
    },
    "step_4b_misogyny_type": {
      "reasoning": "step-by-step reasoning about negative framing of women",
      "type": "shaming | stereotype | objectification | violence | non_misogynistic | unclear"
    },
    "step_4c_text_image_relation": {
      "reasoning": "step-by-step reasoning about text-image interaction",
      "relation": "text_describes_image | text_contradicts_image | text_recontextualizes_image | text_independent_of_image | no_text"
    },
    "step_4d_implicit_associations": {
      "reasoning": "step-by-step reasoning about emergent cross-modal meaning",
      "association": "mechanism explanation, or none"
    }
  }
}"""

# ============================================================
# STEP 2 GUIDELINES — Misogyny (binary), FINALIZED
# Provenance: Chakravarthi et al. 2025 (class anchor, humor/irony),
# Fersini et al. 2022 (MAMI taxonomy), Pan et al. 2025 (U-CoT+
# guideline structure: Implicitness / Tone & Intent / Exception).
# ============================================================
CLASSIFICATION_GUIDELINES = """
## Classification Guidelines: Misogyny / Non_Misogyny

You are classifying a Chinese internet meme into exactly ONE of two
classes, based on structured evidence extracted in a prior analysis step
(scene graph, OCR text, and a Hate Semantic Layer). Apply the following
definitions and decision procedure. These definitions follow the
dataset's characterization (Chakravarthi et al., 2025), the MAMI
annotation taxonomy (Fersini et al., 2022, SemEval-2022 Task 5), and
general harmful meme guideline-crafting principles (Pan et al., 2025,
U-CoT+).

### Class Definitions

**Misogyny**: Following the dataset's characterization (Chakravarthi et
al., 2025), misogynistic memes express gender-based prejudice toward
women, often through culturally grounded visual cues and colloquial
language, and frequently trivialize violence against women and
discriminatory attitudes by employing humor and irony. Operationally
(per the MAMI taxonomy, Fersini et al., 2022), this includes content
that shames, stereotypes, objectifies, or endorses/depicts violence
against women, or otherwise expresses hostility, contempt, or
degradation toward women on the basis of gender. This includes:
body-shaming or slut-shaming, reinforcing demeaning gender stereotypes
(domestic role, intellectual inferiority, promiscuity, manipulation),
sexual objectification without consent or context, dehumanizing
language (animalizing, commodifying), and jokes or "observations" whose
humor depends on treating women as inferior, untrustworthy, or existing
primarily for male benefit/judgment.

**Non_Misogyny**: Any of the following:
- Content unrelated to women or gender topics entirely.
- Content that references women or gender topics neutrally, factually,
  or informatively, without demeaning framing.
- Content that is affirming, celebratory, or genuinely humorous without
  reliance on gender-based degradation.
- Content raising a genuinely neutral question or social/political
  discussion about gender-related topics WITHOUT relying on shaming,
  stereotyping, objectification, or violence framing to make its point.
- Content where the evidence is genuinely insufficient to support a
  Misogyny judgment (default to this class when unsure — do not infer
  hate from ambiguous or absent evidence).

### Guiding Principles (apply throughout, not just as a final check)

- **Implicitness**: Misogynistic content is often implicit — wrapped in
  humor, "just joking" framing, or seemingly neutral visuals. Do not
  require explicit slurs; assess whether the overall construction
  (text + image + coded language) is deliberately designed to demean
  women, even if no single element is overtly hostile on its own.
- **Tone & Intent**: Misogynistic memes frequently trivialize violence
  and discrimination through humor and irony (Chakravarthi et al.,
  2025). Do not default to assuming a meme is harmless just because it
  appears humorous, playful, or lighthearted. Evaluate tone neutrally
  rather than charitably (Pan et al., 2025).
- **Exception**: Do not sacrifice precision for recall. In-group,
  self-referential, or clearly satirical content that mocks misogyny
  itself (rather than women) should default to Non_Misogyny.

### Decision Procedure (apply in order)

1. **Check relevance**: If the scene graph, OCR text, and Hate Semantic
   Layer show no connection to women or gender topics at all ->
   Non_Misogyny. Stop.

2. **Check valence**: Based on the coded language analysis, the
   text-image relationship, and the implicit associations described in
   Step 1, determine whether the content constructs a NEGATIVE
   (shaming / stereotyping / objectifying / violent / degrading)
   framing of women, or a NEUTRAL/POSITIVE one.
   - If neutral or positive -> Non_Misogyny. Stop.
   - If negative -> Misogyny.

3. **Record subtype** (for interpretability, not part of the binary
   decision): use the `misogyny_type` field from Step 1
   (shaming / stereotype / objectification / violence) as supporting
   evidence in your reasoning. If Step 1 marked it "unclear" but your
   own valence check in step 2 is clearly negative, note this
   discrepancy explicitly in your reasoning field rather than silently
   overriding it.

### Important Instructions
- Base your judgment ONLY on the evidence provided (scene graph, OCR
  text, Hate Semantic Layer fields, and the image). Do not assume
  additional context not present in the evidence.
- Follow the decision procedure explicitly and show your reasoning for
  each step before giving the final label.
- Output ONLY a valid, complete JSON object. No markdown fences.
"""

STEP2_PROMPT_TEMPLATE = """You are an expert academic annotator classifying Chinese internet memes for a misogyny detection research study. You are given the image and a structured analysis (scene graph + Hate Semantic Layer) produced in a prior step. Apply the classification guidelines below with explicit Chain-of-Thought reasoning to produce the FINAL classification.

{guidelines}

## Structured Evidence from Step 1 (Scene Graph + Hate Semantic Layer)
```json
{step1_evidence}
```

## Required Output JSON Format
{{
  "step1_relevance_check": {{
    "reasoning": "Apply Decision Procedure Step 1 here.",
    "is_gender_relevant": true
  }},
  "step2_valence_check": {{
    "reasoning": "Apply Decision Procedure Step 2 here, using the coded language, text-image relation, and implicit associations evidence.",
    "valence": "negative | neutral_or_positive"
  }},
  "step3_subtype_note": {{
    "reasoning": "Reference the misogyny_type field from Step 1 and note any discrepancy with your own valence judgment.",
    "misogyny_type": "shaming | stereotype | objectification | violence | non_misogynistic | unclear"
  }},
  "final_label": "Misogyny | Non_Misogyny",
  "confidence": "high | medium | low"
}}

Output ONLY the JSON object above, fully filled in. No markdown fences, no extra text."""


# ============================================================
# SAMPLING (fixed seed, persisted to disk for reproducibility)
# ============================================================
def get_pilot_sample():
    if os.path.exists(SAMPLE_LIST_PATH):
        with open(SAMPLE_LIST_PATH, "r", encoding="utf-8") as f:
            sample = json.load(f)
        print(f"Loaded existing pilot sample list ({len(sample)} items) — reusing for comparability.")
        return sample

    df = pd.read_csv(TEST_CSV)
    # only keep rows whose image actually exists on disk
    df = df[df["filename"].apply(lambda x: os.path.exists(os.path.join(TEST_IMAGE_DIR, str(x))))]

    pos = df[df["label"] == 1]["filename"].tolist()  # Misogyny
    neg = df[df["label"] == 0]["filename"].tolist()  # Non_Misogyny

    rng = random.Random(SEED)
    pos_sample = rng.sample(pos, min(N_PER_CLASS, len(pos)))
    neg_sample = rng.sample(neg, min(N_PER_CLASS, len(neg)))

    sample = (
        [{"filename": f, "gt": "Misogyny"} for f in pos_sample]
        + [{"filename": f, "gt": "Non_Misogyny"} for f in neg_sample]
    )
    with open(SAMPLE_LIST_PATH, "w", encoding="utf-8") as f:
        json.dump(sample, f, indent=2, ensure_ascii=False)
    print(f"Sampled {len(pos_sample)} Misogyny + {len(neg_sample)} Non_Misogyny "
          f"(seed={SEED}). Saved to {SAMPLE_LIST_PATH}")
    return sample


# ============================================================
# GEMINI CALL (shared retry + JSON-parse logic)
# ============================================================
def strip_fences(raw):
    cleaned = raw
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    return cleaned.strip()


def call_gemini_json(client, image_path, prompt, thinking_budget, max_output_tokens):
    """Returns (parsed_json, success, raw_text)."""
    with open(image_path, "rb") as f:
        image_data = f.read()
    ext = os.path.splitext(image_path)[1].lower()
    mime_type = "image/png" if ext == ".png" else "image/gif" if ext == ".gif" else "image/jpeg"

    raw = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[
                    types.Part.from_bytes(data=image_data, mime_type=mime_type),
                    prompt,
                ],
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=max_output_tokens,
                    thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget),
                ),
            )
            raw = response.text.strip()
            parsed = json.loads(strip_fences(raw))
            return parsed, True, raw

        except json.JSONDecodeError:
            if attempt < MAX_RETRIES:
                print(f"    JSON parse failed (attempt {attempt}/{MAX_RETRIES}), retrying in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
            else:
                return None, False, raw
        except Exception as e:
            msg = str(e)
            if "503" in msg or "UNAVAILABLE" in msg or "429" in msg:
                wait = RETRY_DELAY * attempt
                print(f"    Server error (attempt {attempt}/{MAX_RETRIES}), waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    NON-RETRYABLE ERROR: {msg[:300]}")
                return None, False, msg
    return None, False, "Max retries exceeded"


# ============================================================
# STEP 1 RUNNERS (V2 no-CoT: thinking OFF; V3 CoT: thinking capped)
# ============================================================
def run_step1(client, sample, version):
    if version == "v2":
        out_path, prompt = V2_STEP1_PATH, PROMPT_V2_SG_NO_COT
        thinking, max_tok = 0, 4096
    else:
        out_path, prompt = V3_STEP1_PATH, PROMPT_V3_SG_COT
        thinking, max_tok = 1024, 8192

    # checkpoint/resume
    if os.path.exists(out_path):
        with open(out_path, "r", encoding="utf-8") as f:
            results = json.load(f)
        done = {r["filename"] for r in results}
        print(f"[Step1-{version.upper()}] resuming: {len(done)} done")
    else:
        results, done = [], set()

    for item in sample:
        fname, gt = item["filename"], item["gt"]
        if fname in done:
            continue
        image_path = os.path.join(TEST_IMAGE_DIR, fname)

        print(f"[Step1-{version.upper()}] {fname} (GT: {gt})")
        parsed, ok, raw = call_gemini_json(client, image_path, prompt, thinking, max_tok)

        if ok:
            parsed["filename"] = fname
            parsed["ground_truth"] = gt
            results.append(parsed)
            hsl = parsed.get("hate_semantic_layer", {})
            if version == "v2":
                print(f"    type={hsl.get('misogyny_type','N/A')} | "
                      f"coded={[t.get('term') for t in hsl.get('coded_language', [])]}")
            else:
                print(f"    type={hsl.get('step_4b_misogyny_type',{}).get('type','N/A')} | "
                      f"coded={[t.get('term') for t in hsl.get('step_4a_coded_language',{}).get('coded_terms', [])]}")
        else:
            results.append({"filename": fname, "ground_truth": gt,
                            "step1_error": True, "raw_output": raw[:500]})
            print("    FAILED")

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        time.sleep(4)

    print(f"[Step1-{version.upper()}] done -> {out_path}\n")
    return {r["filename"]: r for r in results if not r.get("step1_error")}


# ============================================================
# NORMALIZE STEP 1 OUTPUT (V2 flat vs V3 nested, mirroring HM)
# ============================================================
def normalize_hsl(step1_output, version):
    hsl = step1_output.get("hate_semantic_layer", {})
    if version == "v3":
        coded = hsl.get("step_4a_coded_language", {})
        mtype = hsl.get("step_4b_misogyny_type", {})
        relation = hsl.get("step_4c_text_image_relation", {})
        assoc = hsl.get("step_4d_implicit_associations", {})
        return {
            "coded_language_terms": coded.get("coded_terms", []),
            "coded_language_reasoning": coded.get("reasoning", ""),
            "misogyny_type": mtype.get("type", "unclear"),
            "misogyny_type_reasoning": mtype.get("reasoning", ""),
            "text_image_relation": relation.get("relation", "unclear"),
            "text_image_relation_reasoning": relation.get("reasoning", ""),
            "implicit_associations": assoc.get("association", "none"),
            "implicit_associations_reasoning": assoc.get("reasoning", ""),
        }
    else:  # v2 flat
        return {
            "coded_language_terms": hsl.get("coded_language", []),
            "coded_language_reasoning": "N/A (V2 has no explicit reasoning field)",
            "misogyny_type": hsl.get("misogyny_type", "unclear"),
            "misogyny_type_reasoning": "N/A (V2 has no explicit reasoning field)",
            "text_image_relation": hsl.get("text_image_relation", "unclear"),
            "text_image_relation_reasoning": "N/A (V2 has no explicit reasoning field)",
            "implicit_associations": hsl.get("implicit_associations", "none"),
            "implicit_associations_reasoning": "N/A (V2 has no explicit reasoning field)",
        }


def build_evidence(step1_output, version):
    return {
        "scene_graph": step1_output.get("scene_graph", {}),
        "ocr_text": step1_output.get("ocr_text", ""),
        "hate_semantic_layer": normalize_hsl(step1_output, version),
    }


# ============================================================
# STEP 2 RUNNER
# ============================================================
def run_step2(client, sample, v2_results, v3_results):
    if os.path.exists(STEP2_PATH):
        with open(STEP2_PATH, "r", encoding="utf-8") as f:
            all_results = json.load(f)
        done = {r["filename"] for r in all_results}
        print(f"[Step2] resuming: {len(done)} done")
    else:
        all_results, done = [], set()

    for item in sample:
        fname, gt = item["filename"], item["gt"]
        if fname in done:
            continue
        image_path = os.path.join(TEST_IMAGE_DIR, fname)
        result = {"filename": fname, "ground_truth": gt}
        print(f"[Step2] {fname} (GT: {gt})")

        for version, step1_map in (("v2", v2_results), ("v3", v3_results)):
            if fname not in step1_map:
                result[f"{version}_final_label"] = "N/A (Step1 missing/failed)"
                continue
            prompt = STEP2_PROMPT_TEMPLATE.format(
                guidelines=CLASSIFICATION_GUIDELINES,
                step1_evidence=json.dumps(build_evidence(step1_map[fname], version),
                                          indent=2, ensure_ascii=False),
            )
            parsed, ok, raw = call_gemini_json(client, image_path, prompt,
                                               thinking_budget=1024, max_output_tokens=2048)
            if ok:
                result[f"{version}_final_label"] = parsed.get("final_label", "PARSE_ERROR")
                result[f"{version}_confidence"] = parsed.get("confidence", "N/A")
                result[f"{version}_step2_raw"] = parsed
                print(f"    {version.upper()}: {result[f'{version}_final_label']}")
            else:
                result[f"{version}_final_label"] = "ERROR"
                print(f"    {version.upper()}: FAILED")
            time.sleep(4)

        all_results.append(result)
        with open(STEP2_PATH, "w", encoding="utf-8") as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)

    return all_results


# ============================================================
# SUMMARY
# ============================================================
def summarize(all_results):
    print(f"\n{'='*72}")
    print("MISOGYNY PILOT SUMMARY (20 samples)")
    print(f"{'='*72}")
    print(f"{'Image':<24} | {'GT':<13} | {'V2':<13} | {'V3':<13}")
    print("-" * 72)

    stats = {"v2": [0, 0], "v3": [0, 0]}  # correct, total
    valid = ("Misogyny", "Non_Misogyny")

    for r in all_results:
        gt = r["ground_truth"]
        row = f"{r['filename'][:22]:<24} | {gt:<13}"
        for v in ("v2", "v3"):
            label = r.get(f"{v}_final_label", "N/A")
            mark = ""
            if label in valid:
                stats[v][1] += 1
                if label == gt:
                    stats[v][0] += 1
                    mark = "+"
                else:
                    mark = "x"
            row += f" | {label[:11]:<11}{mark}"
        print(row)

    print("-" * 72)
    for v in ("v2", "v3"):
        c, t = stats[v]
        print(f"{v.upper()} accuracy: {c}/{t}" if t else f"{v.upper()} accuracy: N/A")
    print(f"\nSaved: {STEP2_PATH}")
    print("NOTE: 20-sample accuracy is a sanity check, NOT a reportable result.")
    print("Also check for: refusals, UNKNOWN parses, and 'unclear' misogyny_type")
    print("with negative valence (the step3 discrepancy case) before full run.")


# ============================================================
# MAIN
# ============================================================
def main():
    client = genai.Client(api_key=API_KEY)

    sample = get_pilot_sample()
    v2_results = run_step1(client, sample, "v2")
    v3_results = run_step1(client, sample, "v3")
    all_results = run_step2(client, sample, v2_results, v3_results)
    summarize(all_results)


if __name__ == "__main__":
    main()

Sampled 10 Misogyny + 10 Non_Misogyny (seed=42). Saved to /content/drive/MyDrive/misogyny_pilot_sample_list.json
[Step1-V2] 1090.jpg (GT: Misogyny)
    type=stereotype | coded=[]
[Step1-V2] 238.jpg (GT: Misogyny)
    type=unclear | coded=['CNM', '比心情']
[Step1-V2] 1203.jpg (GT: Misogyny)
    type=shaming | coded=[]
[Step1-V2] 251.jpg (GT: Misogyny)
    type=stereotype | coded=['菜', '坑队友']
[Step1-V2] 64.jpg (GT: Misogyny)
    type=violence | coded=['你妈逼', '瞎BB', '往你妈逼里捅黄瓜']
[Step1-V2] 591.jpg (GT: Misogyny)
    type=shaming | coded=['你妈', '操你妈']
[Step1-V2] 366.jpg (GT: Misogyny)
    type=stereotype | coded=['躺赢']
[Step1-V2] 1320.jpg (GT: Misogyny)
    type=non_misogynistic | coded=[]
[Step1-V2] 66.jpg (GT: Misogyny)
    type=non_misogynistic | coded=[]
[Step1-V2] 1031.jpg (GT: Misogyny)
    type=non_misogynistic | coded=['下贱']
[Step1-V2] 922.jpg (GT: Non_Misogyny)
    type=non_misogynistic | coded=['留子']
[Step1-V2] 1430.jpg (GT: Non_Misogyny)
    type=non_misogynistic | coded=[]
[Step1-V

Scence Graph Misogyny 跑全量

In [7]:
"""
Misogyny FULL RUN: Step 1 (V2 & V3) + Step 2 Classification
============================================================
Full-dataset run over the entire test.csv for the Misogyny dataset
(binary: Misogyny / Non_Misogyny), mirroring the HM pipeline structure.
Identical prompts/logic to the validated pilot — only the sampling is
changed from "random 20" to "all rows in test.csv". Checkpoint/resume is
on for every stage, so an interrupted Colab session picks up where it
left off.

  V1 = zero-shot baseline CoT      -> already run (reuse existing results)
  V2 = Scene Graph WITHOUT CoT     -> Step 1 here (thinking OFF)
  V3 = Scene Graph WITH CoT        -> Step 1 here (CoT fields 4a-4d in JSON)
  Step 2 = guideline-based final classification (required for V2 & V3)

Prompt provenance (documented for Methodology):
  - Scene graph skeleton + JSON format ......... CCoT (Mitra et al., 2024) [MT02]
  - Hate Semantic Layer field design ........... M3Hop-CoT (Kumari et al., 2024) [MT08]
  - misogyny_type taxonomy ..................... MAMI (Fersini et al., 2022) [MT09]
  - Chinese coded-language watchlist ........... ToxiCN MM (Lu et al., 2024) [MT10]
  - Guideline structure (Implicitness/Tone/
    Exception) + guided CoT classification ..... U-CoT+ (Pan et al., 2025) [MT06]
  - Class definition anchor + code-mixing note
    + humor/irony trivialization ............... LT-EDI 2025 overview
                                                 (Chakravarthi et al., 2025) [MT11]

Sampling: random 10 Misogyny + 10 Non_Misogyny from test.csv, fixed seed.
The sampled list is saved to disk so re-runs (and later V-variant
comparisons) always use the SAME 20 images.

Run in Google Colab with Google Drive mounted.
"""

import json
import os
import time
import pandas as pd
from google import genai
from google.genai import types

# ============================================================
# CONFIG - MODIFY THESE
# ============================================================
# In Colab, prefer:
#   from google.colab import userdata
#   API_KEY = userdata.get('GEMINI_API_KEY')


TEST_IMAGE_DIR = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
TEST_CSV = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
BASE_OUT = "/content/drive/MyDrive"

MODEL_NAME = "gemini-2.5-flash"
MAX_RETRIES = 3
RETRY_DELAY = 15

# Full-run artifacts (kept separate from the pilot files so nothing is
# overwritten and the two can be compared if needed).
V2_STEP1_PATH = os.path.join(BASE_OUT, "misogyny_FULL_step1_v2_nocot_results.json")
V3_STEP1_PATH = os.path.join(BASE_OUT, "misogyny_FULL_step1_v3_cot_results.json")
STEP2_PATH = os.path.join(BASE_OUT, "misogyny_FULL_step2_classification_results.json")

# ============================================================
# V2 PROMPT: Scene Graph WITHOUT CoT — Misogyny version
# ============================================================
PROMPT_V2_SG_NO_COT = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially:

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer (Extension for Misogyny Research)
Beyond the standard scene graph, extract the following additional fields specifically designed for detecting misogynistic content. This is for academic research purposes. This layer is informed by an entity-object-relationship (EOR) reasoning structure targeting emotional framing, gender-target awareness, and contextual/cultural knowledge (Kumari et al., 2024).

4a. **coded_language**: Does the embedded text contain slang, homophones, puns, internet slang, or metaphors related to gender or women? List each one with:
   - "term": the original word/phrase
   - "literal_meaning": literal meaning
   - "coded_meaning": potential derogatory or hidden meaning in Chinese internet culture targeting women. If no hidden meaning, output "none".

   Common Chinese coded terms to watch for include but are not limited to:
   田园女权 (fake/entitled feminist slur), 绿茶(婊) (scheming "innocent" woman slur),
   心机婊/圣母婊/婚驴 (manipulative/self-righteous/marriage-livestock slurs),
   舔狗 (used to mock men who over-please women, sometimes flipped to blame women),
   女拳 (mocking "women's fist" — derogatory term for feminists),
   捞女 (gold-digger slur), 母/雌 + derogatory suffix (animalizing women),
   彩礼 framed as extortion, references to women's appearance/body as the
   sole basis of worth, and other stereotyping or dehumanizing shorthand.

   Also watch for Chinese-English CODE-MIXED expressions (common on
   Chinese social media, e.g., English abbreviations or words inserted
   into Chinese sentences), which may themselves carry coded
   derogatory meanings toward women.

   If there is no coded language, output an empty list [].

4b. **misogyny_type**: If the content is negative toward women, which subtype best applies? Choose ONE (based on the MAMI/SemEval-2022 Task 5 taxonomy, Fersini et al. 2022):
   - "shaming" (body-shaming, slut-shaming, mocking appearance/behavior),
   - "stereotype" (reinforcing traditional/domestic/intellectual stereotypes),
   - "objectification" (reducing women to sexual objects or property),
   - "violence" (depicting, joking about, or endorsing violence against women),
   - "non_misogynistic", "unclear"

4c. **text_image_relation**: What is the semantic relationship between the overlaid text and the visual content? Choose ONE:
   - "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text"

4d. **implicit_associations**: Does the combination of text and image potentially trigger negative associations related to women or gender roles? If yes, briefly explain the mechanism. If no, output "none".

## Important Instructions:
- Stay objective. Only extract and describe information. Do NOT make a harmful/harmless classification judgment.
- Even if the content appears sensitive, describe it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {
        "entity": "entity_name",
        "attributes": [
          {"attribute": "attribute_type", "value": "attribute_value"}
        ]
      }
    ],
    "relationships": [
      {
        "subject": "entity_1",
        "relationship": "relation_type",
        "object": "entity_2",
        "reasoning": "why this relationship exists"
      }
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "coded_language": [
      {"term": "...", "literal_meaning": "...", "coded_meaning": "..."}
    ],
    "misogyny_type": "...",
    "text_image_relation": "...",
    "implicit_associations": "..."
  }
}"""

# ============================================================
# V3 PROMPT: Scene Graph WITH CoT — Misogyny version
# (same tasks as V2, but every Hate Semantic Layer field carries
#  an explicit "reasoning" step BEFORE its conclusion, mirroring
#  the HM V3 format so Step 2's normalize_hsl() logic is symmetric)
# ============================================================
PROMPT_V3_SG_COT = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially:

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer with Chain-of-Thought (Extension for Misogyny Research)
Beyond the standard scene graph, extract the following additional fields specifically designed for detecting misogynistic content. This is for academic research purposes. This layer is informed by an entity-object-relationship (EOR) reasoning structure targeting emotional framing, gender-target awareness, and contextual/cultural knowledge (Kumari et al., 2024).

For EACH sub-step below, first write out your step-by-step reasoning explicitly, THEN give your conclusion.

4a. **coded_language**: Reason step by step about whether the embedded text contains slang, homophones, puns, internet slang, or metaphors related to gender or women — including deliberately obfuscated forms (homophone substitution, character splitting, in-group slang). Then list each coded term with:
   - "term": the original word/phrase
   - "literal_meaning": literal meaning
   - "coded_meaning": potential derogatory or hidden meaning in Chinese internet culture targeting women. If no hidden meaning, output "none".

   Common Chinese coded terms to watch for include but are not limited to:
   田园女权 (fake/entitled feminist slur), 绿茶(婊) (scheming "innocent" woman slur),
   心机婊/圣母婊/婚驴 (manipulative/self-righteous/marriage-livestock slurs),
   舔狗 (used to mock men who over-please women, sometimes flipped to blame women),
   女拳 (mocking "women's fist" — derogatory term for feminists),
   捞女 (gold-digger slur), 母/雌 + derogatory suffix (animalizing women),
   彩礼 framed as extortion, references to women's appearance/body as the
   sole basis of worth, and other stereotyping or dehumanizing shorthand.

   Also watch for Chinese-English CODE-MIXED expressions (common on
   Chinese social media, e.g., English abbreviations or words inserted
   into Chinese sentences), which may themselves carry coded
   derogatory meanings toward women.

   If there is no coded language, output an empty list [].

4b. **misogyny_type**: Reason step by step about whether the content frames women negatively, and if so which subtype best applies. Choose ONE (based on the MAMI/SemEval-2022 Task 5 taxonomy, Fersini et al. 2022):
   - "shaming" (body-shaming, slut-shaming, mocking appearance/behavior),
   - "stereotype" (reinforcing traditional/domestic/intellectual stereotypes),
   - "objectification" (reducing women to sexual objects or property),
   - "violence" (depicting, joking about, or endorsing violence against women),
   - "non_misogynistic", "unclear"

4c. **text_image_relation**: Reason step by step about how the overlaid text semantically interacts with the visual content. Choose ONE:
   - "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text"

4d. **implicit_associations**: Reason step by step about whether the COMBINATION of text and image triggers negative associations related to women or gender roles that neither modality carries alone. If yes, briefly explain the mechanism. If no, output "none".

## Important Instructions:
- Stay objective. Only extract, reason, and describe. Do NOT make a final harmful/harmless classification judgment.
- Even if the content appears sensitive, describe it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {
        "entity": "entity_name",
        "attributes": [
          {"attribute": "attribute_type", "value": "attribute_value"}
        ]
      }
    ],
    "relationships": [
      {
        "subject": "entity_1",
        "relationship": "relation_type",
        "object": "entity_2",
        "reasoning": "why this relationship exists"
      }
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "step_4a_coded_language": {
      "reasoning": "step-by-step reasoning about obfuscation and coded terms",
      "coded_terms": [
        {"term": "...", "literal_meaning": "...", "coded_meaning": "..."}
      ]
    },
    "step_4b_misogyny_type": {
      "reasoning": "step-by-step reasoning about negative framing of women",
      "type": "shaming | stereotype | objectification | violence | non_misogynistic | unclear"
    },
    "step_4c_text_image_relation": {
      "reasoning": "step-by-step reasoning about text-image interaction",
      "relation": "text_describes_image | text_contradicts_image | text_recontextualizes_image | text_independent_of_image | no_text"
    },
    "step_4d_implicit_associations": {
      "reasoning": "step-by-step reasoning about emergent cross-modal meaning",
      "association": "mechanism explanation, or none"
    }
  }
}"""

# ============================================================
# STEP 2 GUIDELINES — Misogyny (binary), FINALIZED
# Provenance: Chakravarthi et al. 2025 (class anchor, humor/irony),
# Fersini et al. 2022 (MAMI taxonomy), Pan et al. 2025 (U-CoT+
# guideline structure: Implicitness / Tone & Intent / Exception).
# ============================================================
CLASSIFICATION_GUIDELINES = """
## Classification Guidelines: Misogyny / Non_Misogyny

You are classifying a Chinese internet meme into exactly ONE of two
classes, based on structured evidence extracted in a prior analysis step
(scene graph, OCR text, and a Hate Semantic Layer). Apply the following
definitions and decision procedure. These definitions follow the
dataset's characterization (Chakravarthi et al., 2025), the MAMI
annotation taxonomy (Fersini et al., 2022, SemEval-2022 Task 5), and
general harmful meme guideline-crafting principles (Pan et al., 2025,
U-CoT+).

### Class Definitions

**Misogyny**: Following the dataset's characterization (Chakravarthi et
al., 2025), misogynistic memes express gender-based prejudice toward
women, often through culturally grounded visual cues and colloquial
language, and frequently trivialize violence against women and
discriminatory attitudes by employing humor and irony. Operationally
(per the MAMI taxonomy, Fersini et al., 2022), this includes content
that shames, stereotypes, objectifies, or endorses/depicts violence
against women, or otherwise expresses hostility, contempt, or
degradation toward women on the basis of gender. This includes:
body-shaming or slut-shaming, reinforcing demeaning gender stereotypes
(domestic role, intellectual inferiority, promiscuity, manipulation),
sexual objectification without consent or context, dehumanizing
language (animalizing, commodifying), and jokes or "observations" whose
humor depends on treating women as inferior, untrustworthy, or existing
primarily for male benefit/judgment.

**Non_Misogyny**: Any of the following:
- Content unrelated to women or gender topics entirely.
- Content that references women or gender topics neutrally, factually,
  or informatively, without demeaning framing.
- Content that is affirming, celebratory, or genuinely humorous without
  reliance on gender-based degradation.
- Content raising a genuinely neutral question or social/political
  discussion about gender-related topics WITHOUT relying on shaming,
  stereotyping, objectification, or violence framing to make its point.
- Content where the evidence is genuinely insufficient to support a
  Misogyny judgment (default to this class when unsure — do not infer
  hate from ambiguous or absent evidence).

### Guiding Principles (apply throughout, not just as a final check)

- **Implicitness**: Misogynistic content is often implicit — wrapped in
  humor, "just joking" framing, or seemingly neutral visuals. Do not
  require explicit slurs; assess whether the overall construction
  (text + image + coded language) is deliberately designed to demean
  women, even if no single element is overtly hostile on its own.
- **Tone & Intent**: Misogynistic memes frequently trivialize violence
  and discrimination through humor and irony (Chakravarthi et al.,
  2025). Do not default to assuming a meme is harmless just because it
  appears humorous, playful, or lighthearted. Evaluate tone neutrally
  rather than charitably (Pan et al., 2025).
- **Exception**: Do not sacrifice precision for recall. In-group,
  self-referential, or clearly satirical content that mocks misogyny
  itself (rather than women) should default to Non_Misogyny.

### Decision Procedure (apply in order)

1. **Check relevance**: If the scene graph, OCR text, and Hate Semantic
   Layer show no connection to women or gender topics at all ->
   Non_Misogyny. Stop.

2. **Check valence**: Based on the coded language analysis, the
   text-image relationship, and the implicit associations described in
   Step 1, determine whether the content constructs a NEGATIVE
   (shaming / stereotyping / objectifying / violent / degrading)
   framing of women, or a NEUTRAL/POSITIVE one.
   - If neutral or positive -> Non_Misogyny. Stop.
   - If negative -> Misogyny.

3. **Record subtype** (for interpretability, not part of the binary
   decision): use the `misogyny_type` field from Step 1
   (shaming / stereotype / objectification / violence) as supporting
   evidence in your reasoning. If Step 1 marked it "unclear" but your
   own valence check in step 2 is clearly negative, note this
   discrepancy explicitly in your reasoning field rather than silently
   overriding it.

### Important Instructions
- Base your judgment ONLY on the evidence provided (scene graph, OCR
  text, Hate Semantic Layer fields, and the image). Do not assume
  additional context not present in the evidence.
- Follow the decision procedure explicitly and show your reasoning for
  each step before giving the final label.
- Output ONLY a valid, complete JSON object. No markdown fences.
"""

STEP2_PROMPT_TEMPLATE = """You are an expert academic annotator classifying Chinese internet memes for a misogyny detection research study. You are given the image and a structured analysis (scene graph + Hate Semantic Layer) produced in a prior step. Apply the classification guidelines below with explicit Chain-of-Thought reasoning to produce the FINAL classification.

{guidelines}

## Structured Evidence from Step 1 (Scene Graph + Hate Semantic Layer)
```json
{step1_evidence}
```

## Required Output JSON Format
{{
  "step1_relevance_check": {{
    "reasoning": "Apply Decision Procedure Step 1 here.",
    "is_gender_relevant": true
  }},
  "step2_valence_check": {{
    "reasoning": "Apply Decision Procedure Step 2 here, using the coded language, text-image relation, and implicit associations evidence.",
    "valence": "negative | neutral_or_positive"
  }},
  "step3_subtype_note": {{
    "reasoning": "Reference the misogyny_type field from Step 1 and note any discrepancy with your own valence judgment.",
    "misogyny_type": "shaming | stereotype | objectification | violence | non_misogynistic | unclear"
  }},
  "final_label": "Misogyny | Non_Misogyny",
  "confidence": "high | medium | low"
}}

Output ONLY the JSON object above, fully filled in. No markdown fences, no extra text."""


# ============================================================
# FULL DATASET LOADER (every row in test.csv, in file order)
# ============================================================
def get_full_dataset():
    df = pd.read_csv(TEST_CSV)

    # keep only rows whose image actually exists on disk
    before = len(df)
    df = df[df["filename"].apply(lambda x: os.path.exists(os.path.join(TEST_IMAGE_DIR, str(x))))]
    missing = before - len(df)
    if missing:
        print(f"WARNING: {missing} rows skipped (image file not found on disk).")

    sample = [
        {"filename": str(row["filename"]),
         "gt": "Misogyny" if int(row["label"]) == 1 else "Non_Misogyny"}
        for _, row in df.iterrows()
    ]
    n_pos = sum(1 for s in sample if s["gt"] == "Misogyny")
    n_neg = len(sample) - n_pos
    print(f"Full test set: {len(sample)} images "
          f"({n_pos} Misogyny / {n_neg} Non_Misogyny).")
    return sample


# ============================================================
# GEMINI CALL (shared retry + JSON-parse logic)
# ============================================================
def strip_fences(raw):
    cleaned = raw
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    return cleaned.strip()


def call_gemini_json(client, image_path, prompt, thinking_budget, max_output_tokens):
    """Returns (parsed_json, success, raw_text)."""
    with open(image_path, "rb") as f:
        image_data = f.read()
    ext = os.path.splitext(image_path)[1].lower()
    mime_type = "image/png" if ext == ".png" else "image/gif" if ext == ".gif" else "image/jpeg"

    raw = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[
                    types.Part.from_bytes(data=image_data, mime_type=mime_type),
                    prompt,
                ],
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=max_output_tokens,
                    thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget),
                ),
            )
            raw = response.text.strip()
            parsed = json.loads(strip_fences(raw))
            return parsed, True, raw

        except json.JSONDecodeError:
            if attempt < MAX_RETRIES:
                print(f"    JSON parse failed (attempt {attempt}/{MAX_RETRIES}), retrying in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)
            else:
                return None, False, raw
        except Exception as e:
            msg = str(e)
            if "503" in msg or "UNAVAILABLE" in msg or "429" in msg:
                wait = RETRY_DELAY * attempt
                print(f"    Server error (attempt {attempt}/{MAX_RETRIES}), waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    NON-RETRYABLE ERROR: {msg[:300]}")
                return None, False, msg
    return None, False, "Max retries exceeded"


# ============================================================
# STEP 1 RUNNERS (V2 no-CoT: thinking OFF; V3 CoT: thinking capped)
# ============================================================
def run_step1(client, sample, version):
    if version == "v2":
        out_path, prompt = V2_STEP1_PATH, PROMPT_V2_SG_NO_COT
        thinking, max_tok = 0, 4096
    else:
        out_path, prompt = V3_STEP1_PATH, PROMPT_V3_SG_COT
        thinking, max_tok = 1024, 8192

    # checkpoint/resume
    if os.path.exists(out_path):
        with open(out_path, "r", encoding="utf-8") as f:
            results = json.load(f)
        done = {r["filename"] for r in results}
        print(f"[Step1-{version.upper()}] resuming: {len(done)} done")
    else:
        results, done = [], set()

    for item in sample:
        fname, gt = item["filename"], item["gt"]
        if fname in done:
            continue
        image_path = os.path.join(TEST_IMAGE_DIR, fname)

        print(f"[Step1-{version.upper()}] {fname} (GT: {gt})")
        parsed, ok, raw = call_gemini_json(client, image_path, prompt, thinking, max_tok)

        if ok:
            parsed["filename"] = fname
            parsed["ground_truth"] = gt
            results.append(parsed)
            hsl = parsed.get("hate_semantic_layer", {})
            if version == "v2":
                print(f"    type={hsl.get('misogyny_type','N/A')} | "
                      f"coded={[t.get('term') for t in hsl.get('coded_language', [])]}")
            else:
                print(f"    type={hsl.get('step_4b_misogyny_type',{}).get('type','N/A')} | "
                      f"coded={[t.get('term') for t in hsl.get('step_4a_coded_language',{}).get('coded_terms', [])]}")
        else:
            results.append({"filename": fname, "ground_truth": gt,
                            "step1_error": True, "raw_output": raw[:500]})
            print("    FAILED")

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        time.sleep(4)

    print(f"[Step1-{version.upper()}] done -> {out_path}\n")
    return {r["filename"]: r for r in results if not r.get("step1_error")}


# ============================================================
# NORMALIZE STEP 1 OUTPUT (V2 flat vs V3 nested, mirroring HM)
# ============================================================
def normalize_hsl(step1_output, version):
    hsl = step1_output.get("hate_semantic_layer", {})
    if version == "v3":
        coded = hsl.get("step_4a_coded_language", {})
        mtype = hsl.get("step_4b_misogyny_type", {})
        relation = hsl.get("step_4c_text_image_relation", {})
        assoc = hsl.get("step_4d_implicit_associations", {})
        return {
            "coded_language_terms": coded.get("coded_terms", []),
            "coded_language_reasoning": coded.get("reasoning", ""),
            "misogyny_type": mtype.get("type", "unclear"),
            "misogyny_type_reasoning": mtype.get("reasoning", ""),
            "text_image_relation": relation.get("relation", "unclear"),
            "text_image_relation_reasoning": relation.get("reasoning", ""),
            "implicit_associations": assoc.get("association", "none"),
            "implicit_associations_reasoning": assoc.get("reasoning", ""),
        }
    else:  # v2 flat
        return {
            "coded_language_terms": hsl.get("coded_language", []),
            "coded_language_reasoning": "N/A (V2 has no explicit reasoning field)",
            "misogyny_type": hsl.get("misogyny_type", "unclear"),
            "misogyny_type_reasoning": "N/A (V2 has no explicit reasoning field)",
            "text_image_relation": hsl.get("text_image_relation", "unclear"),
            "text_image_relation_reasoning": "N/A (V2 has no explicit reasoning field)",
            "implicit_associations": hsl.get("implicit_associations", "none"),
            "implicit_associations_reasoning": "N/A (V2 has no explicit reasoning field)",
        }


def build_evidence(step1_output, version):
    return {
        "scene_graph": step1_output.get("scene_graph", {}),
        "ocr_text": step1_output.get("ocr_text", ""),
        "hate_semantic_layer": normalize_hsl(step1_output, version),
    }


# ============================================================
# STEP 2 RUNNER
# ============================================================
def run_step2(client, sample, v2_results, v3_results):
    if os.path.exists(STEP2_PATH):
        with open(STEP2_PATH, "r", encoding="utf-8") as f:
            all_results = json.load(f)
        done = {r["filename"] for r in all_results}
        print(f"[Step2] resuming: {len(done)} done")
    else:
        all_results, done = [], set()

    for item in sample:
        fname, gt = item["filename"], item["gt"]
        if fname in done:
            continue
        image_path = os.path.join(TEST_IMAGE_DIR, fname)
        result = {"filename": fname, "ground_truth": gt}
        print(f"[Step2] {fname} (GT: {gt})")

        for version, step1_map in (("v2", v2_results), ("v3", v3_results)):
            if fname not in step1_map:
                result[f"{version}_final_label"] = "N/A (Step1 missing/failed)"
                continue
            prompt = STEP2_PROMPT_TEMPLATE.format(
                guidelines=CLASSIFICATION_GUIDELINES,
                step1_evidence=json.dumps(build_evidence(step1_map[fname], version),
                                          indent=2, ensure_ascii=False),
            )
            parsed, ok, raw = call_gemini_json(client, image_path, prompt,
                                               thinking_budget=1024, max_output_tokens=2048)
            if ok:
                result[f"{version}_final_label"] = parsed.get("final_label", "PARSE_ERROR")
                result[f"{version}_confidence"] = parsed.get("confidence", "N/A")
                result[f"{version}_step2_raw"] = parsed
                print(f"    {version.upper()}: {result[f'{version}_final_label']}")
            else:
                result[f"{version}_final_label"] = "ERROR"
                print(f"    {version.upper()}: FAILED")
            time.sleep(4)

        all_results.append(result)
        with open(STEP2_PATH, "w", encoding="utf-8") as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)

    return all_results


# ============================================================
# SUMMARY
# ============================================================
def macro_f1(results, version):
    """Binary macro-F1 over Misogyny/Non_Misogyny (matches the shared-task metric)."""
    valid = ("Misogyny", "Non_Misogyny")
    per = {}
    for cls in valid:
        tp = sum(1 for r in results
                 if r.get(f"{version}_final_label") == cls and r["ground_truth"] == cls)
        fp = sum(1 for r in results
                 if r.get(f"{version}_final_label") == cls and r["ground_truth"] != cls)
        fn = sum(1 for r in results
                 if r.get(f"{version}_final_label") != cls and r["ground_truth"] == cls
                 and r.get(f"{version}_final_label") in valid)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        per[cls] = (prec, rec, f1)
    macro = sum(v[2] for v in per.values()) / len(valid)
    return macro, per


def summarize(all_results):
    print(f"\n{'='*72}")
    print(f"MISOGYNY FULL-RUN SUMMARY ({len(all_results)} samples)")
    print(f"{'='*72}")

    valid = ("Misogyny", "Non_Misogyny")
    stats = {"v2": [0, 0], "v3": [0, 0]}  # correct, total
    unparsed = {"v2": [], "v3": []}

    for r in all_results:
        gt = r["ground_truth"]
        for v in ("v2", "v3"):
            label = r.get(f"{v}_final_label", "N/A")
            if label in valid:
                stats[v][1] += 1
                if label == gt:
                    stats[v][0] += 1
            else:
                unparsed[v].append((r["filename"], label))

    for v in ("v2", "v3"):
        c, t = stats[v]
        acc = c / t if t else 0.0
        mf1, per = macro_f1(all_results, v)
        print(f"\n{v.upper()}:")
        print(f"  Accuracy : {c}/{t} = {acc:.4f}" if t else "  Accuracy : N/A")
        print(f"  Macro-F1 : {mf1:.4f}")
        for cls in valid:
            p, rc, f1 = per[cls]
            print(f"    {cls:<14} P={p:.4f} R={rc:.4f} F1={f1:.4f}")
        if unparsed[v]:
            print(f"  Non-standard outputs ({len(unparsed[v])}): "
                  f"{unparsed[v][:8]}{' ...' if len(unparsed[v])>8 else ''}")

    print(f"\nSaved: {STEP2_PATH}")
    print("\nReminder: Macro-F1 is the primary metric (shared-task standard).")
    print("Compare these against the baseline row for Gemini 2.5-flash")
    print("(Misogyny ZS-CoT MF1 = 0.7730) to see if the scene-graph method")
    print("beats the zero-shot baseline. Also inspect any non-standard outputs")
    print("(refusals / parse failures) before treating results as final.")


# ============================================================
# MAIN
# ============================================================
def main():
    client = genai.Client(api_key=API_KEY)

    sample = get_full_dataset()
    v2_results = run_step1(client, sample, "v2")
    v3_results = run_step1(client, sample, "v3")
    all_results = run_step2(client, sample, v2_results, v3_results)
    summarize(all_results)


if __name__ == "__main__":
    main()

Full test set: 340 images (104 Misogyny / 236 Non_Misogyny).
[Step1-V2] resuming: 336 done
[Step1-V2] 991.jpg (GT: Non_Misogyny)
    JSON parse failed (attempt 1/3), retrying in 15s...
    JSON parse failed (attempt 2/3), retrying in 15s...
    FAILED
[Step1-V2] 428.jpg (GT: Misogyny)
    JSON parse failed (attempt 1/3), retrying in 15s...
    JSON parse failed (attempt 2/3), retrying in 15s...
    FAILED
[Step1-V2] 1645.jpg (GT: Non_Misogyny)
    type=non_misogynistic | coded=[]
[Step1-V2] 1408.jpg (GT: Misogyny)
    type=non_misogynistic | coded=[]
[Step1-V2] done -> /content/drive/MyDrive/misogyny_FULL_step1_v2_nocot_results.json

[Step1-V3] resuming: 340 done
[Step1-V3] done -> /content/drive/MyDrive/misogyny_FULL_step1_v3_cot_results.json

[Step2] resuming: 335 done
[Step2] 991.jpg (GT: Non_Misogyny)
    V3: Non_Misogyny
[Step2] 428.jpg (GT: Misogyny)
    V3: Misogyny
[Step2] 68.jpg (GT: Misogyny)
    V2: Non_Misogyny
    V3: Non_Misogyny
[Step2] 1645.jpg (GT: Non_Misogyny)
    V2

清理失败记录From V2 Result

In [6]:
import json

V2_PATH = "/content/drive/MyDrive/misogyny_FULL_step1_v2_nocot_results.json"

with open(V2_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

before = len(results)
# 只保留成功的记录，删掉带 step1_error 的失败记录
results = [r for r in results if not r.get("step1_error")]
after = len(results)

with open(V2_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"清理前 {before} 条，清理后 {after} 条，删除了 {before - after} 条失败记录")

清理前 340 条，清理后 336 条，删除了 4 条失败记录


清理step2的结果文件

In [5]:
STEP2_PATH = "/content/drive/MyDrive/misogyny_FULL_step2_classification_results.json"

with open(STEP2_PATH, "r", encoding="utf-8") as f:
    step2 = json.load(f)

before = len(step2)
# 删掉 v2 或 v3 任一为 N/A 的记录，让它们重新分类
step2 = [r for r in step2
         if r.get("v2_final_label") not in ("N/A (Step1 missing/failed)", "ERROR")
         and r.get("v3_final_label") not in ("N/A (Step1 missing/failed)", "ERROR")]
after = len(step2)

with open(STEP2_PATH, "w", encoding="utf-8") as f:
    json.dump(step2, f, indent=2, ensure_ascii=False)

print(f"Step2清理前 {before} 条，清理后 {after} 条，删除了 {before - after} 条")

Step2清理前 340 条，清理后 335 条，删除了 5 条


In [8]:
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

STEP2_PATH = "/content/drive/MyDrive/misogyny_FULL_step2_classification_results.json"
with open(STEP2_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

valid = ("Misogyny", "Non_Misogyny")
for version in ("v2", "v3"):
    y_true, y_pred = [], []
    for r in results:
        pred = r.get(f"{version}_final_label")
        if pred in valid:
            y_true.append(r["ground_truth"])
            y_pred.append(pred)
    acc = accuracy_score(y_true, y_pred)
    mp, mr, mf1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    wp, wr, wf1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    print(f"{version.upper()}  n={len(y_true)}  ACC={acc:.4f}  MP={mp:.4f} MR={mr:.4f} MF1={mf1:.4f}  WP={wp:.4f} WR={wr:.4f} WF1={wf1:.4f}")

V2  n=338  ACC=0.8284  MP=0.7996 MR=0.7893 MF1=0.7941  WP=0.8260 WR=0.8284 WF1=0.8269
V3  n=340  ACC=0.8441  MP=0.8238 MR=0.7990 MF1=0.8095  WP=0.8408 WR=0.8441 WF1=0.8410
